# `orbital-game` — 2D RT workflow walkthrough

In-plane relative motion only: `RTState` per vehicle (radial + along-track) under HCW-RT dynamics. Cross-track is dropped — the state is 4-dim `[r, t, ṙ, ṫ]` and the action is 2-dim `[Δv_r, Δv_t]`.

Use this when:
- The scenario lives near the orbital plane and cross-track motion is uninteresting
- You want a smaller state space for faster POMDP planning
- You're prototyping a new policy and don't need 3D fidelity yet

Steps covered:

1. Build a `ScenarioConfig` for HCW-RT dynamics
2. Construct an `OrbitalGameEnv` from the config
3. Define a simple guard policy + null policy-state initializer
4. Run two `rollout`s with different seeds
5. Visualize R-T trajectories
6. Plot reward curves and propellant-mass curves
7. Save and reload one rollout via HDF5

**Setup.** Run with `uv` (Jupyter is not in the project deps; the easiest way is):

```bash
uv run --with jupyter --with ipykernel jupyter notebook examples/workflow_2d_rt.ipynb
```

For the full 3D variant, see `workflow_3d_rtn.ipynb`.

In [ ]:
# Requires the project's dev extras (in particular ipywidgets, used by
# interactive_viewer below). If you haven't already, run from the repo root:
#     uv sync --extra dev
# and then restart this kernel.

# Auto-reload edited library code without a kernel restart. Run this cell
# again after editing any orbital_game module to pick up the changes.
%load_ext autoreload
%autoreload 2

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from types import SimpleNamespace

from orbital_game import OrbitalGameEnv, ScenarioConfig, VehicleParamsSpec, rollout, save_run, load_run
from orbital_game.belief import (
    BeliefRollout,
    EKFBeliefUpdater,
    EKFUniformDefaultInitializer,
    run_belief_rollout,
)
from orbital_game.dynamics.hcw import hcw_rt_step
from orbital_game.env.types import BySide, Side
from orbital_game.observations.range_limited import RangeLimitedObservation
from orbital_game.policies.library import ZeroControl
from orbital_game.reference_orbit import ReferenceOrbitState, mean_motion as ref_mean_motion
from orbital_game.registry import DynamicsKey, StateComponentKey
from orbital_game.sampling.spec import ICSpec
from orbital_game.sampling.side import RelativeEllipse
from orbital_game.sampling.mass import ConstantMass
from orbital_game.viz import (
    RolloutScene,
    interactive_viewer,
    plot_reward_curve,
    plot_reward_diagnostic,
    plot_rollout_mass,
    plot_rollout_panels,
    plot_rollout_rewards,
    plot_rt,
    save_animation,
)


## 1. Build a 2D RT scenario

Three things change relative to the 3D variant:

- `guard_components`/`bandit_components` use `StateComponentKey.RT` (4-dim) instead of `RTN` (6-dim)
- `truth_dynamics` and `planning_dynamics` are set to `DynamicsKey.HCW_RT`
- The IC sampler `RelativeEllipse` automatically projects its (R, T, N, Ṙ, Ṫ, Ṅ) state to 2D `[R, T, Ṙ, Ṫ]` when `RT` is in the components — so cross-track ellipse extent is irrelevant in this scenario

Everything else (reference orbit, vehicle params, dt/horizon) is identical.

In [ ]:
# Same RangeLimitedObservation pattern as in the 3D notebook, but with
# StateComponentKey.RT (4-dim in-plane state) and DynamicsKey.HCW_RT.
proto_cfg = ScenarioConfig(
    n_guards=1,
    n_bandits=1,
    epoch_mjd_utc=60067.0,
    reference_orbit=ReferenceOrbitState(
        position_eci=jnp.array([7000e3, 0.0, 0.0]),
        velocity_eci=jnp.array([0.0, 7.5e3, 0.0]),
    ),
    guard_components=(StateComponentKey.RT, StateComponentKey.MASS),
    bandit_components=(StateComponentKey.RT,),
    guard_params=VehicleParamsSpec(dry_mass_kg=100.0, isp_s=220.0, max_thrust_n=5.0),
    bandit_params=VehicleParamsSpec(dry_mass_kg=50.0, isp_s=200.0, max_thrust_n=2.0),
    ic_sampler=ICSpec(
        guard_sampler=RelativeEllipse(
            radial_ellipse_m=1000.0, phase_rad=0.0, sigma_radial_ellipse_m=10.0,
            mass_sampler=ConstantMass(propellant_mass_kg=10.0),
        ),
        bandit_sampler=RelativeEllipse(
            radial_ellipse_m=1000.0, phase_rad=jnp.pi, sigma_radial_ellipse_m=10.0,
        ),
    ),
    truth_dynamics=DynamicsKey.HCW_RT,
    planning_dynamics=DynamicsKey.HCW_RT,
    dt=10.0,
    max_horizon_s=6000.0,
    seed=0,
)
proto_layout = OrbitalGameEnv(proto_cfg).layout

SENSOR_RANGE_M = 2500.0
guard_obs_fn = RangeLimitedObservation(layout=proto_layout, sensor_range_m=SENSOR_RANGE_M, sigma_range=5.0)
bandit_obs_fn = RangeLimitedObservation(layout=proto_layout, sensor_range_m=SENSOR_RANGE_M, sigma_range=5.0)

cfg = ScenarioConfig(
    n_guards=1,
    n_bandits=1,
    epoch_mjd_utc=60067.0,
    reference_orbit=ReferenceOrbitState(
        position_eci=jnp.array([7000e3, 0.0, 0.0]),
        velocity_eci=jnp.array([0.0, 7.5e3, 0.0]),
    ),
    guard_components=(StateComponentKey.RT, StateComponentKey.MASS),
    bandit_components=(StateComponentKey.RT,),
    guard_params=VehicleParamsSpec(dry_mass_kg=100.0, isp_s=220.0, max_thrust_n=5.0),
    bandit_params=VehicleParamsSpec(dry_mass_kg=50.0, isp_s=200.0, max_thrust_n=2.0),
    ic_sampler=ICSpec(
        guard_sampler=RelativeEllipse(
            radial_ellipse_m=1000.0, phase_rad=0.0, sigma_radial_ellipse_m=10.0,
            mass_sampler=ConstantMass(propellant_mass_kg=10.0),
        ),
        bandit_sampler=RelativeEllipse(
            radial_ellipse_m=1000.0, phase_rad=jnp.pi, sigma_radial_ellipse_m=10.0,
        ),
    ),
    truth_dynamics=DynamicsKey.HCW_RT,
    planning_dynamics=DynamicsKey.HCW_RT,
    guard_observation_fn=guard_obs_fn,
    bandit_observation_fn=bandit_obs_fn,
    dt=10.0,
    max_horizon_s=6000.0,
    seed=0,
)

print(f"max_steps (derived):  {cfg.max_steps}")
print(f"sensor range (m):     {SENSOR_RANGE_M}")
print(f"guard obs fn:         {type(cfg.guard_observation_fn).__name__}")
print(f"bandit obs fn:        {type(cfg.bandit_observation_fn).__name__}")


## 2. Construct the environment

`OrbitalGameEnv` resolves the dynamics, actuator, and pluggables based on the config. Because we picked `HCW_RT`, the bandit policy's action dim is 2 (not 3) and the dynamics step takes a 4-dim state.

In [ ]:
env = OrbitalGameEnv(cfg)
print(f"mean motion (rad/s): {env.mean_motion:.3e}")
print(f"layout flat dim:     {env.layout.flat_dim}")  # 4 RT + 1 mass + 4 bandit RT = 9

## 3. Define a policy and run two rollouts

The default policy commands zero Δv each step — the guard takes no maneuvers, so the rollout shows **pure HCW-RT dynamics** evolving from the sampled IC. Because the IC sits on a bounded relative orbit (`RelativeEllipse` with `delta_a=0`), you should see a closed 2:1 ellipse in the R-T plane: the guard oscillates between `±radial_ellipse_m` radially and `±2·radial_ellipse_m` along-track, tracing the same loop each orbital period. The horizon spans ~2 full orbits so the loop closure is unambiguous.

A commented-out alternative gives the guard small random Gaussian impulses each step — switch to it if you want to see propellant depletion in section 6.

`init_ps` returns `None` — we don't track belief in this demo. Action shape is `(N_guards, 2)` for HCW-RT.

In [ ]:
# Default policy: zero-control — no maneuvers, pure HCW-RT dynamics.
# Action shape is (N_guards, 2) for HCW-RT.
guard_policy = ZeroControl(n_vehicles=cfg.n_guards, action_dim=2)
bandit_policy = ZeroControl(n_vehicles=cfg.n_bandits, action_dim=2)

# Alternative: small random Gaussian impulses each step (m/s). Uncomment to
# enable; the guard will perturb its trajectory and visibly deplete
# propellant. Action shape is (N_guards, 2) for HCW-RT.
# def guard_policy_fn(policy_state, obs, key, t):
#     del obs, t
#     return jax.random.normal(key, (cfg.n_guards, 2)) * 0.5, policy_state
# guard_policy = guard_policy_fn

def init_ps(config, env_state, key):
    del config, env_state, key
    return None

key1, key2 = jax.random.split(jax.random.PRNGKey(42), 2)
traj1 = rollout(env, BySide(guard=guard_policy, bandit=bandit_policy),
                BySide(guard=init_ps, bandit=init_ps), key1, n_steps=cfg.max_steps)
traj2 = rollout(env, BySide(guard=guard_policy, bandit=bandit_policy),
                BySide(guard=init_ps, bandit=init_ps), key2, n_steps=cfg.max_steps)

print(f"rollout 1 episode return:  {float(traj1.sides.guard.reward.sum()):.2f}")
print(f"rollout 2 episode return:  {float(traj2.sides.guard.reward.sum()):.2f}")
print(f"rollout 1 first guard RT pos: {traj1.env_state.guards.rt[0, 0, :2]}")
print(f"rollout 2 first guard RT pos: {traj2.env_state.guards.rt[0, 0, :2]}")

## 3b. Partial observability + EKF belief

Beyond pure dynamics, the framework lets each side use its own
**observation function** (any sensor model satisfying the `ObservationFn`
protocol) and its own **belief representation** (any updater satisfying
`BeliefUpdater`). Here we attach a `RangeLimitedObservation` to both sides
— each observer measures the position of every opposing vehicle that is
within `sensor_range_m` (otherwise the visibility mask zeroes the
measurement). On top of that we wire an **EKF belief** that propagates
each tracked target's mean and covariance under HCW dynamics and corrects
on the gated, noisy position measurements. `run_belief_rollout` runs both
the env step and the belief update inside a single `jax.lax.scan`, so the
returned `belief_history` has a leading time axis ready to be visualized.


In [ ]:
# Per-vehicle dynamics adapter for 2D RT. hcw_rt_step is batched (n, 4); we
# wrap it the same way as in the 3D notebook.
n_motion = float(ref_mean_motion(cfg.reference_orbit))
hcw_params = SimpleNamespace(mean_motion=n_motion)

def per_vehicle_hcw_rt(x, u, dt):
    return hcw_rt_step(x[None, :], u[None, :], hcw_params, dt)[0]

ekf_updater = EKFBeliefUpdater(
    dynamics_fn=per_vehicle_hcw_rt,
    process_noise=jnp.eye(4) * 0.01,
    dt=cfg.dt,
)
ekf_initializer = EKFUniformDefaultInitializer(
    layout=env.layout,
    default_mean=jnp.zeros(4),
    variance_diag=jnp.array([100.0, 100.0, 1.0, 1.0]),
)

belief_rollout = BeliefRollout(
    env=env,
    guard_belief_initializer=ekf_initializer, guard_belief_updater=ekf_updater,
    bandit_belief_initializer=ekf_initializer, bandit_belief_updater=ekf_updater,
)

policies_b = BySide(
    guard=ZeroControl(n_vehicles=cfg.n_guards, action_dim=2),
    bandit=ZeroControl(n_vehicles=cfg.n_bandits, action_dim=2),
)
init_ps_b = BySide(
    guard=lambda c, s, k: None,
    bandit=lambda c, s, k: None,
)

traj_b, belief_history = run_belief_rollout(
    belief_rollout, policies_b, init_ps_b, jax.random.PRNGKey(7), n_steps=cfg.max_steps,
)
print(f"belief mean shape:  {belief_history.guard.mean.shape}  (T, N_obs, N_total, d)")
print(f"belief cov shape:   {belief_history.guard.cov.shape}")


## 4. Visualize the in-plane trajectories

Only `plot_rt` makes sense in 2D — there is no cross-track (N) component, so R-N / T-N / 3D RTN plots don't apply. The trace is the guard's `(r, t)` position over time, with both rollouts overlaid for comparison.

In [ ]:
rt1 = traj1.env_state.guards.rt  # (T, N=1, 4)
rt2 = traj2.env_state.guards.rt

fig, ax = plt.subplots(figsize=(8, 8))
plot_rt(rt1, ax=ax, label="rollout 1")
plot_rt(rt2, ax=ax, label="rollout 2")
ax.legend()
ax.set_title("guard R-T trajectory (in-plane HCW)")
plt.show()

## 5. Reward curves

`DistanceToReferenceOrbit` automatically uses 2D distance for RT scenarios — it reads `guards.rt[:, :2]` for position and norms over those two components.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
rewards = jnp.stack([traj1.sides.guard.reward, traj2.sides.guard.reward], axis=0)  # (B=2, T)
plot_reward_curve(rewards, ax=ax)
ax.set_title("reward per step (faint: each rollout, bold: mean)")
plt.show()

## 6. Propellant mass over time

Because the guard's `Mass` component is in `guard_components`, the impulsive actuator deposits a rocket-equation Δm into `propellant_mass` each step — but only when the actuator command is non-zero. With the default zero-control policy, propellant stays at its initial 10 kg (flat line). Uncomment the random-impulse policy in section 3 to see propellant deplete.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(traj1.env_state.guards.propellant_mass[:, 0], label="rollout 1")
ax.plot(traj2.env_state.guards.propellant_mass[:, 0], label="rollout 2")
ax.set_xlabel("step"); ax.set_ylabel("propellant mass (kg)")
ax.set_title("guard propellant over time"); ax.grid(True); ax.legend()
plt.show()

## 7. Persistence — save and reload via HDF5

Save/load works the same regardless of dynamics choice. The flat trajectory dict from `load_run` will have keys like `"env_state.guards.rt"` (instead of `.rtn`) for a 2D scenario.

In [ ]:
import tempfile, pathlib

tmp = pathlib.Path(tempfile.mkdtemp())
out = tmp / "rollout_1.h5"
save_run(out, cfg, traj1)

loaded_cfg, loaded_traj = load_run(out)

print(f"file written:           {out}  ({out.stat().st_size:,} bytes)")
print(f"loaded config matches:  {loaded_cfg.n_guards == cfg.n_guards and loaded_cfg.dt == cfg.dt}")
print(f"trajectory keys:        {sorted(loaded_traj.keys())[:6]} ...")
print(f"reward round-trips:     {bool(jnp.allclose(loaded_traj['sides.guard.reward'], traj1.sides.guard.reward))}")

## 8. Per-rollout multi-actor visualizations + 3D animation

The `viz` module also offers per-rollout views (every guard *and* every
bandit on one figure, color-coded by side) and a configurable 3D
animation system. The animation share a single per-frame renderer between
an interactive `ipywidgets` viewer and an MP4 exporter, with toggleable
layers:

- **`show_cubes`** — translucent cubes for each agent (oriented by
  `Attitude.quat` if the state component is present, identity otherwise)
- **`show_sensor_range`** — translucent grey sphere at each agent showing
  its sensor-range gate; auto-inferred from
  `cfg.guard_observation_fn` / `cfg.bandit_observation_fn` if they are
  `RangeLimitedObservation`
- **`show_thrust`** — Δv arrows in RTN, scaled so the largest maneuver in
  the rollout occupies 1/20 of the largest axis extent
- **`show_belief`** — 2σ position ellipsoids drawn from the EKF belief
  history per (observer, tracked) pair
- **`show_trail`** — fading trail of past positions (truth)

The reward diagnostic plot overlays the *signal driving the reward*
against the recorded reward, so you can confirm by eye that your reward
construction is doing what you intend.


In [ ]:
# All actors of one rollout, color-coded by side.
fig = plot_rollout_panels(traj_b)
plt.show()


In [ ]:
# Reward-construction diagnostic — overlays the signal driving the reward
# against the recorded reward. Game-aware: dispatches per cfg.game type.
fig = plot_reward_diagnostic(traj_b, cfg)
plt.show()


In [ ]:
# Build a scene with all toggles enabled. show_sensor_range will be
# auto-inferred from cfg.guard/bandit_observation_fn (RangeLimitedObservation
# → translucent grey sphere at sensor_range_m). show_belief renders 2σ
# position ellipsoids per (observer, tracked) pair from belief_history.
scene = RolloutScene(
    traj=traj_b,
    cfg=cfg,
    dt=cfg.dt,
    show_cubes=True,
    show_trail=True,
    show_sensor_range=True,
    show_thrust=False,        # zero-control rollout: nothing to draw
    show_belief=True,
    belief_history=belief_history,
    title_prefix="LBG demo  ",
)
print(f"axis limit (m):  {scene.axis_limit_m:.0f}")
print(f"n_frames:        {scene.n_frames}")

# Interactive viewer: ipywidgets Play + IntSlider. Works in classic Jupyter
# and JupyterLab; outside those use save_animation below.
fig_anim = interactive_viewer(scene)


In [ ]:
# Save the same scene to MP4. Requires ffmpeg on PATH. Output goes to
# `<repo_root>/outputs/`, which is gitignored.
import shutil, pathlib

def _repo_root() -> pathlib.Path:
    here = pathlib.Path.cwd().resolve()
    for parent in (here, *here.parents):
        if (parent / "pyproject.toml").exists():
            return parent
    return here

if shutil.which("ffmpeg") is not None:
    out_dir = _repo_root() / "outputs"
    out_dir.mkdir(exist_ok=True)
    out_path = out_dir / "rollout_2d_rt.mp4"
    save_animation(scene, str(out_path), fps=15, dpi=90)
    print(f"saved {out_path}  ({out_path.stat().st_size:,} bytes)")
else:
    print("ffmpeg not available — skipping MP4 export")


## Next steps

Same pluggable points as the 3D notebook, plus the option to upgrade dimensionality once your 2D prototype is working. The mapping to swap a scenario from 2D → 3D is mechanical:

- `StateComponentKey.RT` → `StateComponentKey.RTN`
- `DynamicsKey.HCW_RT` → `DynamicsKey.HCW_RTN`
- Set non-zero `cross_track_m` on `RelativeEllipse` to introduce out-of-plane motion (the same sampler works in both 2D and 3D)
- Policy action shape goes from `(N, 2)` → `(N, 3)`

All the framework pieces (rollout, save/load, reward, termination, plots) work in both regimes.